Downlond 100k

In [2]:
import os, zipfile, urllib.request
from pathlib import Path

DATA_DIR = Path("data_movielens_100k")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = DATA_DIR / "ml-100k.zip"

if not zip_path.exists():
    print("Downloading MovieLens 100K...")
    urllib.request.urlretrieve(URL, zip_path.as_posix())
else:
    print("Zip already downloaded:", zip_path)

# Unzip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

print("Extracted to:", DATA_DIR)
print("Top-level contents:", list(DATA_DIR.iterdir())[:5])

Extracted to: data_movielens_100k
Top-level contents: [WindowsPath('data_movielens_100k/ml-100k'), WindowsPath('data_movielens_100k/ml-100k.zip')]


Load ratings (u.data)

In [23]:
import pandas as pd

ratings_path = DATA_DIR / "ml-100k" / "u.data"
# u.data format: user_id \t item_id \t rating \t timestamp
ratings = pd.read_csv(
    ratings_path,
    sep="\t",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
)

ratings.head(), ratings.shape

(   user_id  item_id  rating  timestamp
 0      196      242       3  881250949
 1      186      302       3  891717742
 2       22      377       1  878887116
 3      244       51       2  880606923
 4      166      346       1  886397596,
 (100000, 4))

Train/test split (random 80/20 over observed ratings)

In [24]:

import numpy as np

rng = np.random.default_rng(42)
idx = np.arange(len(ratings))
rng.shuffle(idx)

split = int(0.8 * len(idx))
train_idx = idx[:split]
test_idx  = idx[split:]

train = ratings.iloc[train_idx].reset_index(drop=True)
test  = ratings.iloc[test_idx].reset_index(drop=True)

print("Train size:", len(train), "Test size:", len(test))
print(train)

Train size: 80000 Test size: 20000
       user_id  item_id  rating  timestamp
0          354       60       5  891217160
1          620        8       3  889988121
2           99      121       3  885679261
3          250      202       4  878090253
4          886       24       4  876031973
...        ...      ...     ...        ...
79995      286      856       2  877533698
79996      297      182       3  875239125
79997      201      513       3  884114069
79998      559      202       1  891035674
79999      678       14       3  879544815

[80000 rows x 4 columns]


Candidate Generator

In [38]:
import random

all_users = set(train['user_id'].unique())

user = random.choice(list(all_users))

M=1000

def get_seen_movies(user_id, ratings):
    return set(ratings[ratings['user_id'] == user_id]['item_id'])

def generate_candidates(user, ratings):
    all_movies = set(ratings['item_id'].unique())
    seen_movies = get_seen_movies(user, ratings)
    
    return list(all_movies - seen_movies)

movies_unreview = generate_candidates(user, train)

def topM(ratings, user, M):
    candidates = generate_candidates(user, ratings)
    return candidates[:M]

movies_unreview = topM(train, user, M)

print(movies_unreview)

[2, 3, 4, 5, 6, 8, 11, 12, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 125, 126, 128, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 141, 142, 143, 144, 145, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 222, 223, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 23

Ranker:

. Popularity baseline:

In [44]:



def popularity_ranker(train_df):
    movie_counts = (
        train_df
        .groupby("item_id")
        .size()
        .reset_index(name="interaction_count")
        .sort_values(by="interaction_count", ascending=False)
    )
    return movie_counts

popularity_df = popularity_ranker(train)

def rank_candidates_by_popularity(popularity_df):
    
    return popularity_df.sort_values(by="interaction_count", ascending=False)['item_id'].tolist()

top_movies = rank_candidates_by_popularity(popularity_df)

print(popularity_df.head())
print(top_movies)

     item_id  interaction_count
49        50                453
99       100                419
257      258                419
180      181                405
285      286                384
[50, 258, 100, 181, 286, 294, 288, 1, 174, 300, 127, 121, 7, 56, 117, 237, 222, 98, 172, 204, 313, 405, 79, 173, 210, 269, 151, 69, 168, 302, 748, 22, 257, 25, 276, 9, 318, 423, 15, 328, 118, 64, 195, 183, 216, 96, 111, 234, 176, 202, 186, 742, 28, 191, 97, 289, 89, 82, 268, 275, 135, 238, 196, 323, 11, 132, 357, 483, 12, 153, 125, 546, 144, 245, 185, 475, 194, 333, 282, 70, 182, 228, 471, 265, 197, 655, 568, 143, 301, 273, 435, 8, 179, 496, 603, 271, 508, 322, 427, 95, 235, 71, 678, 88, 175, 4, 200, 215, 161, 588, 187, 180, 272, 474, 208, 597, 274, 230, 403, 209, 134, 211, 14, 385, 124, 515, 147, 284, 732, 250, 307, 393, 527, 298, 13, 327, 203, 566, 340, 154, 23, 751, 326, 283, 480, 591, 514, 845, 432, 692, 511, 628, 218, 252, 433, 479, 451, 24, 58, 684, 651, 319, 229, 582, 419, 137, 133, 367, 44